# 02 — Feature Engineering

Builds customer (RFM), product (baskets), and time (daily sales) feature tables from the DuckDB warehouse built in Notebook 1.

**Important**: `customer_unique_id` (not `customer_id`) is used for all customer-level aggregation -- Olist generates a new `customer_id` per order, so grouping by `customer_id` would make every repeat customer look like a first-time buyer.

In [1]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb pandas pyarrow scikit-learn xgboost lightgbm mlxtend shap prophet google-genai

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROCESSED_DIR / "features"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse.duckdb"

for d in [PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence


In [2]:

import duckdb
con = duckdb.connect(str(WAREHOUSE_PATH))
print("Connected to warehouse:", WAREHOUSE_PATH)


Connected to warehouse: /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence/warehouse.duckdb


## Customer features (RFM + satisfaction + delivery experience)

In [3]:

import pandas as pd

query = """
SELECT
    c.customer_unique_id, c.customer_state,
    f.order_id, f.price, f.freight_value, f.payment_value, f.payment_installments,
    f.delivery_days, f.delivery_delay_days, d.full_date AS order_date,
    r.review_score, p.category_name_english
FROM fact_order_item f
JOIN dim_customer c ON f.customer_key = c.customer_key
JOIN dim_date d ON f.order_date_key = d.date_key
LEFT JOIN dim_review r ON f.review_key = r.review_key
LEFT JOIN dim_product p ON f.product_key = p.product_key
"""
df = con.execute(query).df()
df["order_date"] = pd.to_datetime(df["order_date"])

order_level = df.groupby(["customer_unique_id", "order_id"]).agg(
    order_merchandise_value=("price", "sum"),
    order_freight=("freight_value", "sum"),
    order_payment=("payment_value", "max"),
    order_date=("order_date", "first"),
    order_installments=("payment_installments", "max"),
    order_delivery_days=("delivery_days", "first"),
    order_delivery_delay=("delivery_delay_days", "first"),
    order_review=("review_score", "first"),
).reset_index()

reference_date = order_level["order_date"].max() + pd.Timedelta(days=1)

customer_features = order_level.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    monetary=("order_payment", "sum"),
    avg_order_value=("order_payment", "mean"),
    avg_freight=("order_freight", "mean"),
    first_purchase=("order_date", "min"),
    last_purchase=("order_date", "max"),
    avg_installments=("order_installments", "mean"),
    avg_delivery_days=("order_delivery_days", "mean"),
    avg_delivery_delay=("order_delivery_delay", "mean"),
    avg_review_score=("order_review", "mean"),
).reset_index()

customer_features["recency_days"] = (reference_date - customer_features["last_purchase"]).dt.days
customer_features["customer_tenure_days"] = (customer_features["last_purchase"] - customer_features["first_purchase"]).dt.days

category_diversity = (df.dropna(subset=["category_name_english"])
                       .groupby("customer_unique_id")["category_name_english"].nunique()
                       .rename("distinct_categories").reset_index())
customer_features = customer_features.merge(category_diversity, on="customer_unique_id", how="left")
customer_features["distinct_categories"] = customer_features["distinct_categories"].fillna(0).astype(int)

state = df.groupby("customer_unique_id")["customer_state"].agg(
    lambda x: x.mode().iat[0] if not x.mode().empty else "unknown").rename("customer_state")
customer_features = customer_features.merge(state, on="customer_unique_id", how="left")

customer_features.to_parquet(FEATURES_DIR / "customer_features.parquet", index=False)
print(f"customer_features: {len(customer_features):,} customers (vs {df['customer_unique_id'].nunique():,} unique -- "
      f"dim_customer has one row per customer_id, this table is deduplicated to real customers)")
customer_features.head()


customer_features: 95,420 customers (vs 95,420 unique -- dim_customer has one row per customer_id, this table is deduplicated to real customers)


,customer_unique_id,frequency,monetary,avg_order_value,avg_freight,first_purchase,last_purchase,avg_installments,avg_delivery_days,avg_delivery_delay,avg_review_score,recency_days,customer_tenure_days,distinct_categories,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,12.00,2018-05-10,2018-05-10,8.0,6.0,-5.0,5.0,117,0,1,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,8.29,2018-05-07,2018-05-07,1.0,3.0,-5.0,4.0,120,0,1,SP
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,17.22,2017-03-10,2017-03-10,8.0,25.0,-2.0,3.0,543,0,1,SC
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,17.63,2017-10-12,2017-10-12,4.0,20.0,-12.0,4.0,327,0,1,PA
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,16.89,2017-11-14,2017-11-14,6.0,13.0,-8.0,5.0,294,0,1,SP


## Product features (order baskets for association rule mining)

In [4]:

basket_query = """
SELECT f.order_id, p.category_name_english
FROM fact_order_item f
JOIN dim_product p ON f.product_key = p.product_key
WHERE p.category_name_english IS NOT NULL AND p.category_name_english != 'unknown'
"""
basket_df = con.execute(basket_query).df()
order_baskets = basket_df[["order_id", "category_name_english"]].drop_duplicates()
order_baskets.to_parquet(FEATURES_DIR / "order_baskets.parquet", index=False)
print(f"order_baskets: {len(order_baskets):,} rows across {order_baskets['order_id'].nunique():,} orders")


order_baskets: 97,997 rows across 97,256 orders


## Time features (daily sales, with lag and rolling-window features)

In [5]:

daily_query = """
SELECT d.full_date AS order_date, f.order_id, f.price
FROM fact_order_item f JOIN dim_date d ON f.order_date_key = d.date_key
"""
daily_raw = con.execute(daily_query).df()
daily_raw["order_date"] = pd.to_datetime(daily_raw["order_date"])

daily_sales = daily_raw.groupby("order_date").agg(
    revenue=("price", "sum"), order_count=("order_id", "nunique")).reset_index().sort_values("order_date")

full_range = pd.date_range(daily_sales["order_date"].min(), daily_sales["order_date"].max(), freq="D")
daily_sales = (daily_sales.set_index("order_date").reindex(full_range).fillna(0.0)
               .rename_axis("order_date").reset_index())

daily_sales["day_of_week"] = daily_sales["order_date"].dt.dayofweek
daily_sales["is_weekend"] = daily_sales["day_of_week"].isin([5, 6])
daily_sales["month"] = daily_sales["order_date"].dt.month
daily_sales["quarter"] = daily_sales["order_date"].dt.quarter

for lag in [1, 7, 30]:
    daily_sales[f"revenue_lag_{lag}"] = daily_sales["revenue"].shift(lag)
    daily_sales[f"order_count_lag_{lag}"] = daily_sales["order_count"].shift(lag)
for window in [7, 30]:
    daily_sales[f"revenue_rolling_mean_{window}"] = daily_sales["revenue"].rolling(window).mean()
    daily_sales[f"order_count_rolling_mean_{window}"] = daily_sales["order_count"].rolling(window).mean()

daily_sales.to_parquet(FEATURES_DIR / "daily_sales.parquet", index=False)
con.close()
print(f"daily_sales: {len(daily_sales):,} days")
daily_sales.tail()


daily_sales: 730 days


,order_date,revenue,order_count,day_of_week,is_weekend,month,quarter,revenue_lag_1,order_count_lag_1,revenue_lag_7,order_count_lag_7,revenue_lag_30,order_count_lag_30,revenue_rolling_mean_7,order_count_rolling_mean_7,revenue_rolling_mean_30,order_count_rolling_mean_30
725,2018-08-30,0.0,0.0,3,False,8,3,1546.04,11.0,17030.87,142.0,45316.28,320.0,5552.798571,50.857143,28489.544333,215.066667
726,2018-08-31,0.0,0.0,4,False,8,3,0.00,0.0,9633.61,98.0,40553.80,310.0,4176.568571,36.857143,27137.751000,204.733333
727,2018-09-01,0.0,0.0,5,True,9,3,0.00,0.0,10599.41,69.0,38462.26,299.0,2662.367143,27.000000,25855.675667,194.766667
728,2018-09-02,0.0,0.0,6,True,9,3,0.00,0.0,8070.71,73.0,42075.18,314.0,1509.408571,16.571429,24453.169667,184.300000
729,2018-09-03,145.0,1.0,0,False,9,3,0.00,0.0,5345.91,66.0,35511.96,245.0,766.421429,7.285714,23274.271000,176.166667
